# 04 — Distance metrics: Levenshtein, Jaccard, normalised Tree Edit Distance

Where notebook 03 produces *binary* / *categorical* signals (EM, per-component F1), this notebook produces **continuous distances** in `[0, 1]` where `0.0` means identical and `1.0` means maximally different. Continuous signals are useful for ranking, calibration, and detecting near-misses that a binary EM would miss.

Three metrics:

| Metric | Operates on | Cost | Comments |
|---|---|---|---|
| **Levenshtein** (token-level, normalised) | Token stream | Cheap | Sensitive to renaming. We canonicalise first so it's not *that* sensitive. |
| **Jaccard distance** | Token sets | Cheap | Order-insensitive — often a feature for queries. |
| **Normalised Tree Edit Distance (TED)** | Shallow clause tree | Medium (APTED algorithm) | Standard for code similarity. Robust to surface variation once the tree is canonicalised. **This is the headline distance metric for the thesis.** |

All three are computed per record and saved to `metrics_distance.csv`. The final report joins them with the structural and execution metrics on `(query_id, dialect)`.

In [ ]:
from __future__ import annotations

import json
import logging
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import pandas as pd
import yaml

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

OUTPUTS_DIR = REPO_ROOT / "evaluation" / "outputs"
RECORDS_PATH = OUTPUTS_DIR / "records.json"
DATASETS_DIR = REPO_ROOT / "evaluation" / "datasets"
OUT_CSV = OUTPUTS_DIR / "metrics_distance.csv"

Dialect = Literal["cypher", "aql"]
logger = logging.getLogger("distance_metrics")

## Tokeniser + canonicaliser

Same routines as notebook 03 — duplicated here so each notebook is self-contained. If you change canonicalisation, change both places (or extract into a shared module if you find yourself doing this often).

In [ ]:
_STRING_LITERAL = re.compile(r"'(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\"")
_NUMBER_LITERAL = re.compile(r"\b\d+(?:\.\d+)?\b")
_PUNCT = re.compile(r"[(){}\[\],;]|->|<-|<>|>=|<=|==|!=|=|<|>|\.|:|\+|\-|\*|/|\|")
_IDENT = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
_COMMENT_LINE = re.compile(r"//[^\n]*|--[^\n]*")
_COMMENT_BLOCK = re.compile(r"/\*.*?\*/", re.DOTALL)


def strip_comments(query: str) -> str:
    return _COMMENT_LINE.sub(" ", _COMMENT_BLOCK.sub(" ", query))


def tokenize(query: str) -> list[str]:
    text = strip_comments(query)
    tokens: list[str] = []
    i, n = 0, len(text)
    while i < n:
        if text[i].isspace():
            i += 1
            continue
        for pat in (_STRING_LITERAL, _NUMBER_LITERAL, _IDENT, _PUNCT):
            m = pat.match(text, i)
            if m:
                tokens.append(m.group(0))
                i = m.end()
                break
        else:
            tokens.append(text[i])
            i += 1
    return tokens


_CYPHER_KEYWORDS = frozenset([
    "MATCH", "OPTIONAL", "WHERE", "RETURN", "WITH", "ORDER", "BY", "LIMIT", "SKIP",
    "UNWIND", "CALL", "CREATE", "MERGE", "DELETE", "DETACH", "SET", "REMOVE", "UNION",
    "DISTINCT", "AS", "AND", "OR", "NOT", "IN", "IS", "NULL", "TRUE", "FALSE",
    "ASC", "DESC", "CONTAINS", "STARTS", "ENDS", "WHEN", "CASE", "THEN", "ELSE", "END", "FOREACH",
])
_AQL_KEYWORDS = frozenset([
    "FOR", "IN", "LET", "RETURN", "FILTER", "SORT", "LIMIT", "COLLECT",
    "OUTBOUND", "INBOUND", "ANY", "GRAPH", "AGGREGATE", "WITH", "INTO", "DISTINCT",
    "ASC", "DESC", "AND", "OR", "NOT", "TRUE", "FALSE", "NULL",
    "INSERT", "UPDATE", "REPLACE", "REMOVE", "UPSERT", "OPTIONS", "LIKE",
])
_CYPHER_CLAUSE_HEADS = (
    "MATCH", "OPTIONAL MATCH", "WHERE", "RETURN", "WITH", "ORDER BY", "LIMIT", "SKIP",
    "UNWIND", "CALL", "CREATE", "MERGE", "DELETE", "DETACH DELETE", "SET", "REMOVE", "UNION",
)
_AQL_CLAUSE_HEADS = (
    "FOR", "LET", "FILTER", "SORT", "LIMIT", "COLLECT", "RETURN",
    "INSERT", "UPDATE", "REPLACE", "REMOVE", "UPSERT",
)


def keywords_for(d: Dialect) -> frozenset[str]:
    return _CYPHER_KEYWORDS if d == "cypher" else _AQL_KEYWORDS


def clause_heads_for(d: Dialect) -> tuple[str, ...]:
    return _CYPHER_CLAUSE_HEADS if d == "cypher" else _AQL_CLAUSE_HEADS


_CYPHER_BINDING = re.compile(r"[(\[]([A-Za-z_][A-Za-z0-9_]*)\b")
_AQL_BINDING = re.compile(r"\b(?:FOR|LET)\s+([A-Za-z_][A-Za-z0-9_]*)\b", re.IGNORECASE)


def alpha_rename(query: str, bindings: list[str]) -> str:
    rename: dict[str, str] = {}
    for n in bindings:
        if n not in rename:
            rename[n] = f"v{len(rename)}"
    if not rename:
        return query
    pat = re.compile(r"\b(" + "|".join(re.escape(n) for n in sorted(rename, key=len, reverse=True)) + r")\b")
    return pat.sub(lambda m: rename[m.group(1)], query)


def canonicalize(query: str, dialect: Dialect) -> str:
    text = strip_comments(query)
    bindings_re = _CYPHER_BINDING if dialect == "cypher" else _AQL_BINDING
    bindings = [m.group(1) for m in bindings_re.finditer(text)]
    text = alpha_rename(text, bindings)
    kws = keywords_for(dialect)
    out: list[str] = []
    for t in tokenize(text):
        if t.startswith("'") or t.startswith('"'):
            out.append(t)
        else:
            up = t.upper()
            out.append(up if up in kws else t)
    return " ".join(out)

## Levenshtein (token-level, normalised)

Classic Wagner-Fischer edit distance on the canonical token stream, normalised by the longer token list. `normalised = 0` ⇔ identical token sequences.

In [ ]:
def _levenshtein_tokens(a: list[str], b: list[str]) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    curr = [0] * (len(b) + 1)
    for i, ai in enumerate(a, start=1):
        curr[0] = i
        for j, bj in enumerate(b, start=1):
            cost = 0 if ai == bj else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev, curr = curr, prev
    return prev[-1]


def levenshtein(translated: str, expected: str, dialect: Dialect) -> float:
    ta = canonicalize(translated, dialect).split(" ")
    tb = canonicalize(expected, dialect).split(" ")
    denom = max(len(ta), len(tb))
    return _levenshtein_tokens(ta, tb) / denom if denom else 0.0

## Jaccard (token-set distance)

`1 - |A ∩ B| / |A ∪ B|` over the canonical token sets. Order-insensitive.

In [ ]:
def jaccard(translated: str, expected: str, dialect: Dialect) -> float:
    ta = set(canonicalize(translated, dialect).split(" "))
    tb = set(canonicalize(expected, dialect).split(" "))
    if not ta and not tb:
        return 0.0
    union = len(ta | tb)
    if union == 0:
        return 0.0
    return 1.0 - len(ta & tb) / union

## Tree Edit Distance (APTED on a shallow clause tree)

We build a two-level tree per query:

```
ROOT (label="QUERY")
├── CLAUSE 1  (label="MATCH")
│   ├── token leaf
│   ├── token leaf
│   └── ...
├── CLAUSE 2  (label="WHERE")
│   └── ...
└── ...
```

APTED computes the optimal sequence of node insertions/deletions/relabels to turn one tree into the other. We then normalise by the larger tree's size.

The tree is intentionally shallow — full openCypher AST parsing is out of scope for an evaluation harness, and a clause-level tree already captures clause presence/order/contents, which is most of what matters for query similarity.

In [ ]:
from apted import APTED, Config


@dataclass
class ClauseNode:
    label: str
    children: list["ClauseNode"] = field(default_factory=list)

    def size(self) -> int:
        return 1 + sum(c.size() for c in self.children)


def _match_clause_head(tokens: list[str], i: int, heads: tuple[str, ...]) -> tuple[str, int] | None:
    # Multi-word heads ("OPTIONAL MATCH", "ORDER BY") must beat single-word heads.
    for h in sorted(heads, key=lambda x: -len(x.split())):
        parts = h.split()
        if i + len(parts) > len(tokens):
            continue
        if all(tokens[i + k].upper() == parts[k] for k in range(len(parts))):
            return h, len(parts)
    return None


def parse_to_clause_tree(query: str, dialect: Dialect) -> ClauseNode:
    tokens = canonicalize(query, dialect).split(" ")
    heads = clause_heads_for(dialect)
    root = ClauseNode(label="QUERY")
    current: ClauseNode | None = None
    i = 0
    while i < len(tokens):
        m = _match_clause_head(tokens, i, heads)
        if m is not None:
            label, consumed = m
            current = ClauseNode(label=label)
            root.children.append(current)
            i += consumed
            continue
        if current is None:
            current = ClauseNode(label="<prelude>")
            root.children.append(current)
        current.children.append(ClauseNode(label=tokens[i]))
        i += 1
    return root


class _TreeConfig(Config):
    def rename(self, n1: ClauseNode, n2: ClauseNode) -> int:
        return 0 if n1.label == n2.label else 1

    def children(self, node: ClauseNode) -> list[ClauseNode]:
        return node.children


def normalized_ted(translated: str, expected: str, dialect: Dialect) -> float:
    t_tree = parse_to_clause_tree(translated, dialect)
    e_tree = parse_to_clause_tree(expected, dialect)
    distance = APTED(t_tree, e_tree, _TreeConfig()).compute_edit_distance()
    denom = max(t_tree.size(), e_tree.size())
    return distance / denom if denom else 0.0

## Identity sanity test

Feed each gold query as its own translation; all three distances should be 0.0. Any non-zero result means the canonicalisation or tree builder is non-deterministic and our numbers are unreliable.

In [ ]:
identity_rows = []
for ds_path in sorted(DATASETS_DIR.glob("*.yaml")):
    data = yaml.safe_load(ds_path.read_text())
    for q in data["queries"]:
        for dialect, key in (("cypher", "expected_cypher"), ("aql", "expected_aql")):
            ref = q[key]
            identity_rows.append((
                q["id"], dialect,
                levenshtein(ref, ref, dialect),
                jaccard(ref, ref, dialect),
                normalized_ted(ref, ref, dialect),
            ))
identity_df = pd.DataFrame(identity_rows, columns=["query_id", "dialect", "lev", "jac", "ted"])
non_zero = identity_df[(identity_df[["lev", "jac", "ted"]] > 1e-9).any(axis=1)]
print(f"Identity test: {len(identity_df)} cases; non-zero distances: {len(non_zero)}")
if len(non_zero):
    display(non_zero)
    raise AssertionError("Identity sanity test failed — a distance metric has a bug.")
print("PASS")

## Compute on the real records

In [ ]:
records = json.loads(RECORDS_PATH.read_text())
rows = []
for r in records:
    translated = r.get("generated_query")
    expected = r["expected_query"]
    dialect = r["dialect"]
    if not translated:
        rows.append({
            "query_id": r["query_id"], "schema_name": r["schema_name"], "dialect": dialect,
            "difficulty": r["difficulty"],
            "levenshtein": 1.0, "jaccard": 1.0, "normalized_ted": 1.0,
        })
        continue
    rows.append({
        "query_id": r["query_id"], "schema_name": r["schema_name"], "dialect": dialect,
        "difficulty": r["difficulty"],
        "levenshtein": levenshtein(translated, expected, dialect),
        "jaccard": jaccard(translated, expected, dialect),
        "normalized_ted": normalized_ted(translated, expected, dialect),
    })
dist_df = pd.DataFrame(rows)
dist_df.head(8)

## Summaries (lower is better)

In [ ]:
metric_cols = ["levenshtein", "jaccard", "normalized_ted"]
print("Overall (mean):")
display(dist_df[metric_cols].mean().to_frame("mean"))
print("\nBy schema × dialect:")
display(dist_df.groupby(["schema_name", "dialect"])[metric_cols].mean())
print("\nBy difficulty:")
display(dist_df.groupby("difficulty")[metric_cols].mean().reindex(["easy", "medium", "hard"]))

In [ ]:
dist_df.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(dist_df)} rows to {OUT_CSV}")